In [ ]:
pip install requests beautifulsoup4

In [ ]:
pip install selenium webdriver-manager

In [1]:
import pandas as pd
import time
import random
from bs4 import BeautifulSoup

# --- Import các thư viện Selenium cần thiết ---
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# --- PHẦN CẤU HÌNH ---
INPUT_FILE = r'D:\ai002\archive\male_players_final.csv'
OUTPUT_FILE = r'D:\ai002\archive\male_players_final.csv'
URL_COLUMN = 'url'
SAVE_EVERY = 500 # Giảm xuống để lưu thường xuyên hơn, vì Selenium chạy chậm hơn

def get_career_with_selenium(driver, player_url):
    """
    Hàm này sử dụng Selenium để tải trang, đợi JavaScript chạy,
    sau đó lấy HTML đã được render hoàn chỉnh để phân tích.
    """
    try:
        driver.get(player_url)

        # === ĐÂY LÀ BƯỚC QUAN TRỌNG NHẤT ===
        # Đợi tối đa 10 giây cho đến khi tiêu đề 'Career' xuất hiện.
        # Điều này đảm bảo trang đã tải xong và vượt qua các bài kiểm tra.
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//h5[text()='Career']"))
        )

        # Lấy toàn bộ HTML của trang sau khi đã tải xong
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'html.parser')
        
        teams = set()
        # Phần logic phân tích HTML bên dưới giữ nguyên như cũ
        career_heading = soup.find('h5', string='Career')
        if career_heading:
            career_table = career_heading.find_next_sibling('table')
            if career_table:
                team_links = career_table.find_all('a', href=lambda href: href and '/team/' in href)
                for link in team_links:
                    team_name = link.text.strip()
                    if team_name:
                        teams.add(team_name)
        
        player_meta = soup.find('div', class_='meta')
        if player_meta:
            national_team_link = player_meta.find('a', href=lambda href: href and '/teams?na=' in href)
            if national_team_link:
                national_team_name = national_team_link.text.strip()
                if national_team_name:
                    teams.add(national_team_name)

        return list(teams)

    except Exception as e:
        print(f"Lỗi khi xử lý URL với Selenium: {player_url} - {e}")
        return [] # Trả về rỗng nếu có lỗi

# --- PHẦN XỬ LÝ CHÍNH ĐÃ ĐƯỢC CẬP NHẬT ---
try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file '{INPUT_FILE}'.")
    exit()

if 'team_color' not in df.columns:
    df['team_color'] = pd.NA

# --- Khởi tạo trình duyệt Selenium MỘT LẦN duy nhất ---
print("Đang khởi tạo trình duyệt Chrome...")
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless")  # Chạy ở chế độ nền, không mở cửa sổ
chrome_options.add_argument("--log-level=3") # Giảm bớt log không cần thiết
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36")
service = ChromeService(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
print("Trình duyệt đã sẵn sàng.")

total_rows = len(df)
for index, row in df.iterrows():
    if pd.notna(row.get('team_color')) and row.get('team_color') != '':
        continue

    player_url = row[URL_COLUMN]
    if pd.isna(player_url) or not str(player_url).startswith('http'):
        continue
    
    player_name = row.get('long_name', f'Cầu thủ ở hàng {index + 1}')
    print(f"Đang xử lý ({index + 1}/{total_rows}): {player_name}...")
    
    # Gọi hàm Selenium
    team_history = get_career_with_selenium(driver, player_url)
    
    if team_history:
        team_color_str = ', '.join(sorted(team_history))
        df.at[index, 'team_color'] = team_color_str
        print(f"-> Đã tìm thấy: {team_color_str}")
    else:
        print("-> Không tìm thấy dữ liệu.")
        
    if (index + 1) % SAVE_EVERY == 0:
        df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
        print(f"--- Đã lưu tiến trình tại hàng {index + 1} ---")

    time.sleep(random.uniform(1, 3)) # Vẫn giữ độ trễ nhỏ

# --- Đóng trình duyệt sau khi hoàn tất ---
driver.quit()
print("Đã đóng trình duyệt.")

df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"\nHoàn tất! Dữ liệu đã được lưu vào file '{OUTPUT_FILE}'.")

Đang khởi tạo trình duyệt Chrome...
Trình duyệt đã sẵn sàng.
Đang xử lý (15001/16161): Cầu thủ ở hàng 15001...
-> Đã tìm thấy: Beijing Guoan, Changchun Yatai, Notts County, Wolverhampton Wanderers
Đang xử lý (15002/16161): Cầu thủ ở hàng 15002...
-> Đã tìm thấy: Viktoria Köln
Đang xử lý (15003/16161): Cầu thủ ở hàng 15003...
-> Đã tìm thấy: Accrington Stanley
Đang xử lý (15004/16161): Cầu thủ ở hàng 15004...
-> Đã tìm thấy: Galway United
Đang xử lý (15005/16161): Cầu thủ ở hàng 15005...
-> Đã tìm thấy: DC United
Đang xử lý (15006/16161): Cầu thủ ở hàng 15006...
-> Đã tìm thấy: Damac FC
Đang xử lý (15007/16161): Cầu thủ ở hàng 15007...
-> Đã tìm thấy: Hartlepool United, Rotherham United
Đang xử lý (15008/16161): Cầu thủ ở hàng 15008...
-> Đã tìm thấy: Sønderjyske Fodbold
Đang xử lý (15009/16161): Cầu thủ ở hàng 15009...
-> Đã tìm thấy: Kristiansund BK
Đang xử lý (15010/16161): Cầu thủ ở hàng 15010...
-> Đã tìm thấy: Chengdu Rongcheng, Qingdao West Coast FC
Đang xử lý (15011/16161): Cầu 

In [1]:
import pandas as pd

# --- PHẦN BẠN CẦN KIỂM TRA VÀ CHỈNH SỬA ---

# 1. Đường dẫn đến file CSV của bạn.
# Hãy chắc chắn đường dẫn này là chính xác.
FILE_PATH = r'D:\ai002\archive\male_players_final.csv' 
# Nếu file CSV và file script này nằm cùng thư mục, bạn có thể chỉ cần ghi:
# FILE_PATH = 'male_players_final.csv'

# 2. Tên của cột chứa URL của SoFIFA. 
# Hãy kiểm tra file CSV của bạn để đảm bảo tên cột này là chính xác.
URL_COLUMN_NAME = 'url' 

# 3. Tên của cột ID mới mà bạn muốn tạo.
NEW_ID_COLUMN_NAME = 'ID'

# --------------------------------------------------


def extract_id_from_url(url):
    """
    Hàm này nhận vào một URL từ SoFIFA và trích xuất ra phần ID ở cuối.
    Ví dụ: 'https://sofifa.com/player/231747' -> '231747'
    """
    # Xử lý trường hợp ô dữ liệu trống hoặc không phải là chuỗi
    if not isinstance(url, str) or not url.strip():
        return None
    
    try:
        # Tách chuỗi bằng dấu '/' từ bên phải, chỉ tách 1 lần và lấy phần tử cuối cùng
        player_id = url.rsplit('/', 1)[-1]
        
        # Kiểm tra xem ID có phải là số không để đảm bảo tính hợp lệ
        if player_id.isdigit():
            return int(player_id) # Chuyển thành kiểu số nguyên
        else:
            return None
    except IndexError:
        # Xử lý trường hợp URL không có định dạng đúng
        return None

# --- PHẦN XỬ LÝ CHÍNH ---

print(f"Bắt đầu quá trình xử lý file: '{FILE_PATH}'")

try:
    df = pd.read_csv(FILE_PATH)
    print("1. Đã đọc file CSV thành công.")
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file tại '{FILE_PATH}'. Vui lòng kiểm tra lại đường dẫn.")
    exit()
except Exception as e:
    print(f"Đã có lỗi xảy ra khi đọc file: {e}")
    exit()

# Kiểm tra xem cột URL có tồn tại không
if URL_COLUMN_NAME not in df.columns:
    print(f"Lỗi: Không tìm thấy cột URL có tên '{URL_COLUMN_NAME}' trong file.")
    print(f"Các cột hiện có là: {list(df.columns)}")
    exit()
    
# Kiểm tra xem cột ID đã tồn tại chưa để tránh tạo lại
if NEW_ID_COLUMN_NAME in df.columns:
    print(f"Cảnh báo: Cột '{NEW_ID_COLUMN_NAME}' đã tồn tại. Sẽ tiến hành ghi đè lên cột này.")

# Bước chính: Áp dụng hàm trích xuất ID cho toàn bộ cột URL và tạo cột mới
print(f"2. Đang trích xuất ID từ cột '{URL_COLUMN_NAME}' để tạo cột '{NEW_ID_COLUMN_NAME}'...")
df[NEW_ID_COLUMN_NAME] = df[URL_COLUMN_NAME].apply(extract_id_from_url)
print("   -> Trích xuất hoàn tất.")

# Sắp xếp lại các cột để đưa cột ID mới lên đầu (tùy chọn, nhưng nên làm cho dễ nhìn)
print("3. Đang sắp xếp lại các cột...")
# Lấy danh sách tất cả các cột
all_columns = list(df.columns)
# Xóa cột ID mới ra khỏi danh sách
all_columns.remove(NEW_ID_COLUMN_NAME)
# Chèn cột ID mới vào vị trí đầu tiên
new_column_order = [NEW_ID_COLUMN_NAME] + all_columns
df = df[new_column_order]
print("   -> Sắp xếp hoàn tất.")


# Lưu lại file
print(f"4. Đang lưu những thay đổi lại vào file '{FILE_PATH}'...")
print("   CẢNH BÁO: Thao tác này sẽ GHI ĐÈ lên file gốc của bạn.")
try:
    # index=False để không lưu thêm một cột index không cần thiết vào file CSV
    df.to_csv(FILE_PATH, index=False, encoding='utf-8-sig')
    print("\nQUÁ TRÌNH HOÀN TẤT!")
    print(f"File '{FILE_PATH}' đã được cập nhật thành công với cột '{NEW_ID_COLUMN_NAME}' mới.")
except Exception as e:
    print(f"Lỗi: Không thể lưu file. Chi tiết lỗi: {e}")

Bắt đầu quá trình xử lý file: 'D:\ai002\archive\male_players_final.csv'
1. Đã đọc file CSV thành công.
2. Đang trích xuất ID từ cột 'url' để tạo cột 'ID'...
   -> Trích xuất hoàn tất.
3. Đang sắp xếp lại các cột...
   -> Sắp xếp hoàn tất.
4. Đang lưu những thay đổi lại vào file 'D:\ai002\archive\male_players_final.csv'...
   CẢNH BÁO: Thao tác này sẽ GHI ĐÈ lên file gốc của bạn.

QUÁ TRÌNH HOÀN TẤT!
File 'D:\ai002\archive\male_players_final.csv' đã được cập nhật thành công với cột 'ID' mới.


In [4]:
import os
try:
    from docx import Document
    from docx.shared import Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_BREAK
except ImportError:
    print("Bạn cần cài thư viện python-docx: pip install python-docx")
    exit()

doc = Document()

# --- CẤU HÌNH FORMAT ---
style = doc.styles['Normal']
font = style.font
font.name = 'Times New Roman'
font.size = Pt(12)

def add_heading(text, level):
    h = doc.add_heading(text, level=level)
    h.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = h.runs[0]
    run.font.name = 'Times New Roman'
    run.font.color.rgb = RGBColor(0, 0, 0) # Màu đen

def add_paragraph(text, bold=False, italic=False):
    p = doc.add_paragraph()
    runner = p.add_run(text)
    runner.bold = bold
    runner.italic = italic
    runner.font.name = 'Times New Roman'
    return p

def add_math_block(latex_code, description=None):
    """Thêm khối công thức toán học"""
    if description:
        p_desc = doc.add_paragraph(description)
        p_desc.italic = True
    
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    runner = p.add_run(latex_code)
    runner.font.name = 'Cambria Math' # Font chuyên dụng cho toán
    runner.font.color.rgb = RGBColor(0, 50, 150) # Xanh đậm để nổi bật
    runner.font.size = Pt(12)
    runner.italic = True

# --- BẮT ĐẦU NỘI DUNG BÁO CÁO ---

# TITLE
title = doc.add_heading('BÁO CÁO KHOA HỌC: CƠ SỞ TOÁN HỌC CỦA SỰ CÔNG BẰNG TRONG HỌC MÁY\n(MATHEMATICAL FOUNDATIONS OF FAIRNESS IN MACHINE LEARNING)', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

add_paragraph('Tóm tắt:', bold=True)
add_paragraph('Báo cáo này phân tích nguồn gốc của thiên kiến thuật toán, xây dựng cơ sở toán học cho bài toán công bằng từ các nguyên lý cơ bản (MLE, Gradient Descent), và đề xuất phương pháp tối ưu hóa ràng buộc sử dụng nhân tử Lagrange. Báo cáo cũng trình bày chứng minh toán học cho "Định lý Bất khả thi" trong việc thỏa mãn đa tiêu chí.')

doc.add_paragraph().add_run().add_break(WD_BREAK.PAGE) # Ngắt trang

# I. GIỚI THIỆU
add_heading('I. GIỚI THIỆU CHUNG (INTRODUCTION)', level=1)
add_paragraph('1. Bối cảnh:', bold=True)
add_paragraph('Trí tuệ nhân tạo (AI) đang định hình lại các quyết định quan trọng trong xã hội. Tuy nhiên, các mô hình học máy truyền thống thường hoạt động như những "hộp đen" tối ưu hóa độ chính xác mà bỏ qua các yếu tố đạo đức, dẫn đến việc tái tạo hoặc khuếch đại các định kiến xã hội.')

add_paragraph('2. Vấn đề nghiên cứu:', bold=True)
add_paragraph('Làm thế nào để định lượng khái niệm "công bằng" trừu tượng thành các công thức toán học khả thi? Làm thế nào để tích hợp các ràng buộc này vào quá trình tối ưu hóa mô hình mà không làm giảm quá nhiều hiệu suất dự đoán?')

# II. NGUỒN GỐC THIÊN KIẾN
add_heading('II. NGUỒN GỐC CỦA THIÊN KIẾN (SOURCES OF BIAS)', level=1)
add_paragraph('Sự bất công trong AI không phải lỗi ngẫu nhiên mà xuất phát từ các nguồn gốc cụ thể:')
add_paragraph('• Thiên kiến Dữ liệu (Data Bias):', bold=True)
add_paragraph('  - Historical Bias: Dữ liệu phản ánh các định kiến lịch sử (ví dụ: lương nam giới cao hơn nữ giới trong dữ liệu cũ).')
add_paragraph('  - Representation Bias: Dữ liệu thiếu sự đa dạng (ví dụ: tập dữ liệu ảnh chỉ chứa người da trắng).')
add_paragraph('• Thiên kiến Thuật toán (Algorithmic Bias):', bold=True)
add_paragraph('  - Tối ưu hóa sai lệch: Hàm mất mát (Loss function) truyền thống chỉ quan tâm đến độ chính xác trung bình toàn cục, sẵn sàng hy sinh độ chính xác của nhóm thiểu số.')
add_paragraph('  - Biến thay thế (Proxy Variables): Thuật toán học các biến tương quan (như mã bưu chính) để thay thế cho thuộc tính nhạy cảm đã bị loại bỏ.')

# III. CƠ SỞ TOÁN HỌC
add_heading('III. CƠ SỞ TOÁN HỌC & MÔ HÌNH HÓA (MATHEMATICAL FOUNDATIONS)', level=1)
add_paragraph('Để giải quyết bài toán, ta cần xây dựng lại mô hình học máy từ các nguyên lý cơ bản.')

add_heading('1. Thiết lập bài toán chuẩn (Standard Setup)', level=2)
add_paragraph('Xét không gian dữ liệu D = {(x, a, y)}, trong đó x là vector đặc trưng, a là thuộc tính nhạy cảm (0/1), và y là nhãn thực tế. Ta sử dụng mô hình Hồi quy Logistic để ước lượng xác suất:')
add_math_block(r'h_\theta(x) = \sigma(z) = \frac{1}{1 + e^{-z}}, \quad \text{với } z = \theta^T x + b')

add_heading('2. Hàm mất mát cơ sở (Baseline Loss Function)', level=2)
add_paragraph('Dựa trên nguyên lý Ước lượng hợp lý cực đại (Maximum Likelihood Estimation - MLE), ta tìm tham số theta để tối đa hóa xác suất quan sát dữ liệu. Điều này tương đương với việc cực tiểu hóa hàm Binary Cross-Entropy (BCE):')
add_math_block(r'J_{BCE}(\theta) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \log(h_\theta(x_i)) + (1-y_i) \log(1-h_\theta(x_i)) \right]')

add_heading('3. Định lượng sự Công bằng (Fairness Quantification)', level=2)
add_paragraph('Ta chọn tiêu chuẩn Demographic Parity (DP). Về mặt lý thuyết, ta cần P(Y_hat=1 | A=0) = P(Y_hat=1 | A=1). Tuy nhiên, để đưa vào bài toán tối ưu (cần tính đạo hàm), ta sử dụng phương pháp "nới lỏng" (relaxation) bằng kỳ vọng dự đoán:')
add_math_block(r'\bar{\mu}_a = \mathbb{E}[h_\theta(x) | A=a] \approx \frac{1}{|N_a|} \sum_{i \in N_a} h_\theta(x_i)')
add_paragraph('Hàm phạt sự bất công (Fairness Penalty) được định nghĩa là bình phương sự chênh lệch:')
add_math_block(r'J_{fair}(\theta) = (\bar{\mu}_1 - \bar{\mu}_0)^2')

# IV. TỐI ƯU HÓA
add_heading('IV. CHIẾN LƯỢC TỐI ƯU HÓA (OPTIMIZATION STRATEGY)', level=1)

add_heading('1. Hàm Lagrangian (The Lagrangian)', level=2)
add_paragraph('Bài toán trở thành tối ưu hóa có ràng buộc. Ta sử dụng phương pháp nhân tử Lagrange để chuyển về dạng không ràng buộc:')
add_math_block(r'\mathcal{L}(\theta, \lambda) = J_{BCE}(\theta) + \lambda \cdot J_{fair}(\theta)')
add_paragraph('Trong đó, lambda là siêu tham số kiểm soát sự đánh đổi (trade-off).')

add_heading('2. Đạo hàm & Gradient Descent', level=2)
add_paragraph('Đây là cơ sở toán học cho thuật toán được cài đặt trong phần thực nghiệm. Ta áp dụng quy tắc chuỗi (Chain Rule) để tính Gradient:')
add_math_block(r'\nabla_\theta \mathcal{L} = \nabla_\theta J_{BCE} + \lambda \cdot \nabla_\theta J_{fair}')
add_paragraph('Phân tích thành phần Fairness Gradient:')
add_math_block(r'\frac{\partial J_{fair}}{\partial \theta} = 2(\bar{\mu}_1 - \bar{\mu}_0) \cdot \left( \frac{\partial \bar{\mu}_1}{\partial \theta} - \frac{\partial \bar{\mu}_0}{\partial \theta} \right)')
add_paragraph('Với đạo hàm của trung bình nhóm (sử dụng tính chất đạo hàm Sigmoid):')
add_math_block(r'\frac{\partial \bar{\mu}_a}{\partial \theta} = \frac{1}{|N_a|} \sum_{i \in N_a} \underbrace{h_\theta(x_i)(1 - h_\theta(x_i))}_{\sigma^\prime} \cdot x_i')

# V. CASE STUDIES
add_heading('V. MÔ HÌNH THỰC NGHIỆM & CASE STUDIES', level=1)
add_paragraph('1. Mô hình tự xây dựng (From Scratch):', bold=True)
add_paragraph('Dựa trên các công thức tại phần IV, chúng tôi đã xây dựng class "FairLogisticRegression" bằng Python. Kết quả thực nghiệm cho thấy:')
add_paragraph('- Khi Lambda = 0 (Unfair): Độ chính xác cao nhưng chênh lệch giữa hai nhóm lớn.')
add_paragraph('- Khi Lambda tăng: Chênh lệch giảm mạnh về 0, độ chính xác giảm nhẹ (minh chứng cho sự đánh đổi).')

add_paragraph('2. Các ví dụ thực tế:', bold=True)
add_paragraph('- Hệ thống COMPAS (Mỹ): Dự đoán tái phạm tội thiên kiến chống lại người da màu.')
add_paragraph('- Tuyển dụng Amazon: Loại bỏ ứng viên nữ do dữ liệu lịch sử.')

# VI. THÁCH THỨC
add_heading('VI. THÁCH THỨC & ĐỊNH LÝ BẤT KHẢ THI', level=1)
add_paragraph('Một thách thức nền tảng trong toán học về sự công bằng là chúng ta không thể thỏa mãn mọi tiêu chí cùng lúc.')
add_heading('Chứng minh Định lý Bất khả thi (Impossibility Theorem)', level=2)
add_paragraph('Ta chứng minh rằng không thể thỏa mãn đồng thời Equalized Odds (Cân bằng TPR/FPR) và Predictive Parity (Cân bằng Precision) nếu tỷ lệ cơ bản (Base Rate) khác nhau.')
add_paragraph('Sử dụng định lý Bayes để khai triển Precision (PPV):')
add_math_block(r'PPV = P(Y=1|\hat{Y}=1) = \frac{TPR \cdot p}{TPR \cdot p + FPR \cdot (1-p)}')
add_paragraph('Trong đó p = P(Y=1) là tỷ lệ cơ bản. Nếu TPR và FPR bằng nhau giữa hai nhóm (Equalized Odds) nhưng p khác nhau, thì giá trị PPV bắt buộc phải khác nhau. Do đó, hai tiêu chí này loại trừ lẫn nhau.')

# VII. KẾT LUẬN
add_heading('VII. KẾT LUẬN (CONCLUSION)', level=1)
add_paragraph('Sự công bằng trong AI không chỉ là vấn đề xã hội mà là một bài toán tối ưu hóa đa mục tiêu phức tạp. Việc giải quyết đòi hỏi sự kết hợp giữa tiền xử lý dữ liệu, thiết kế hàm mất mát phù hợp (Lagrangian Relaxation) và sự giám sát của con người (Human-in-the-loop).')

# VIII. TÀI LIỆU THAM KHẢO
add_heading('VIII. TÀI LIỆU THAM KHẢO', level=1)
add_paragraph('1. Hardt, M., Price, E., & Srebro, N. (2016). Equality of opportunity in supervised learning.')
add_paragraph('2. Kleinberg, J., Mullainathan, S., & Raghavan, M. (2016). Inherent trade-offs in the fair determination of risk scores.')
add_paragraph('3. Barocas, S., Hardt, M., & Narayanan, A. Fairness and Machine Learning.')

# LƯU FILE
file_name = 'Bao_Cao_Fairness_Full_Academic.docx'
doc.save(file_name)
print(f"Đã tạo file báo cáo hoàn chỉnh: {os.path.abspath(file_name)}")

Đã tạo file báo cáo hoàn chỉnh: d:\ai002\Bao_Cao_Fairness_Full_Academic.docx
